# CueNote Phase 9E-H Colab Training

? ???? ????? **Run all** ? ? ??? ???? ??.

???? ?? ??? ?? ?? ?? ?? **?? ? ??**???.

1. Colab ???? Runtime -> Change runtime type -> GPU? ????.
2. ?? ?? ??? RUN_MODE? ????.
3. Run all? ???.
4. Google Drive mount ??? ????.

?? RUN_MODE? `SMOKE`?. ?? Layout ??? `LAYOUT_TRAIN`?? ???. Symbol confidence? ?? 0? ????? `SYMBOL_OVERFIT`?? tiny-overfit? ???? ??, full-page overfit? ????? `SYMBOL_TILE_OVERFIT`?? crop/tile ??? ???? ?? `SYMBOL_TILE_TRAIN`? 권장 symbol 학습 경로로 사용한다. 기존 `SYMBOL_TRAIN`은 full-page diagnostic-only 경로다. ?? ??? ?? DeepScoresV2 dense subset, Drive 14GB/?? 3GB ?? ?? ???? ????.

? ???? ?? DeepScoresV2? ???? ???. raw archive, extracted files, training cache, run directory? `/content/cuenote-phase9` scratch? ??, Drive?? manifest, checkpoint, report, ONNX, artifact? ????.

In [ ]:
# ===== User configuration: edit this cell only =====
import os

# GitHub repository to clone into /content/CueNote.
# If your repository/branch is different, change these two values before Run all.
REPO_URL = "https://github.com/YuYoungKwang/CueNote.git"
REPO_BRANCH = "develop"

# SMOKE | LAYOUT_TRAIN | SYMBOL_TRAIN | SYMBOL_OVERFIT | SYMBOL_TILE_OVERFIT | SYMBOL_TILE_TRAIN | FULL_PIPELINE
# Use SYMBOL_TILE_TRAIN for the recommended symbol detector path. SYMBOL_TRAIN remains full-page diagnostic-only.
RUN_MODE = "SMOKE"

# Drive root for long-lived outputs.
DRIVE_ROOT = "/content/drive/MyDrive/CueNote"

# Small first-run limits for free Colab and 14GB Drive.
MAX_DENSE_IMAGES = 64
MAX_DENSE_SOURCE_GROUPS = 10
TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 4
TRAIN_INPUT_SIZE = 1024
EARLY_STOPPING_PATIENCE = 3
TILE_CROP_SIZE = 768  # 512 | 768 | 1024
TILE_OVERLAP = 0.25

if RUN_MODE == "SYMBOL_OVERFIT":
    MAX_DENSE_IMAGES = 70
    MAX_DENSE_SOURCE_GROUPS = 40
    TRAIN_EPOCHS = 100
    TRAIN_BATCH_SIZE = 1
    TRAIN_INPUT_SIZE = 1280
    EARLY_STOPPING_PATIENCE = 0

if RUN_MODE == "SYMBOL_TILE_OVERFIT":
    MAX_DENSE_IMAGES = 70
    MAX_DENSE_SOURCE_GROUPS = 40
    TRAIN_EPOCHS = 100
    TRAIN_BATCH_SIZE = 2
    TRAIN_INPUT_SIZE = TILE_CROP_SIZE
    EARLY_STOPPING_PATIENCE = 0

if RUN_MODE == "SYMBOL_TILE_TRAIN":
    MAX_DENSE_IMAGES = 320
    MAX_DENSE_SOURCE_GROUPS = 40
    TRAIN_EPOCHS = 40
    TRAIN_BATCH_SIZE = 4
    TRAIN_INPUT_SIZE = TILE_CROP_SIZE
    EARLY_STOPPING_PATIENCE = 5

# Keep this false for first-run/reproducible smoke training.
# Set true only when you intentionally want to continue a compatible unfinished checkpoint.
AUTO_RESUME = False

# Keep this true unless you intentionally need to inspect stale /content files.
# The dataset converted subset on Drive is not deleted by this option.
CLEAN_SCRATCH_RUNS = True

os.environ["CUENOTE_REPO_URL"] = REPO_URL
os.environ["CUENOTE_REPO_BRANCH"] = REPO_BRANCH
os.environ["CUENOTE_RUN_MODE"] = RUN_MODE
os.environ["CUENOTE_DRIVE_ROOT"] = DRIVE_ROOT

print("CueNote Phase 9E-H settings")
print("  repo:", REPO_URL)
print("  branch:", REPO_BRANCH)
print("  run mode:", RUN_MODE)
print("  drive root:", DRIVE_ROOT)
print("  subset images:", MAX_DENSE_IMAGES)
print("  source groups:", MAX_DENSE_SOURCE_GROUPS)
print("  epochs:", TRAIN_EPOCHS)
print("  batch:", TRAIN_BATCH_SIZE)
print("  input size:", TRAIN_INPUT_SIZE)

In [ ]:
# ===== Mount Google Drive and check runtime =====
from pathlib import Path
from google.colab import drive
import os
import platform
import shutil

print("Mounting Google Drive...")
drive.mount("/content/drive")

drive_root = Path(os.environ["CUENOTE_DRIVE_ROOT"])
drive_root.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Drive root:", drive_root)
print("/content usage:", shutil.disk_usage("/content"))
print("Drive usage:", shutil.disk_usage(str(drive_root)))

In [ ]:
# ===== Clone a clean repository checkout =====
from pathlib import Path
import os
import shutil
import subprocess

repo_url = os.environ["CUENOTE_REPO_URL"]
repo_branch = os.environ["CUENOTE_REPO_BRANCH"]
repo_dir = Path("/content/CueNote")

# A clean checkout avoids stale hotfix cells or modified files inside Colab.
if repo_dir.exists():
    print("Removing existing /content/CueNote checkout...")
    shutil.rmtree(repo_dir)

print(f"Cloning {repo_url} branch {repo_branch}...")
subprocess.run(["git", "clone", "--branch", repo_branch, "--single-branch", repo_url, str(repo_dir)], check=True)

commit = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repository:", repo_dir)
print("Branch:", subprocess.check_output(["git", "-C", str(repo_dir), "branch", "--show-current"], text=True).strip())
print("Commit:", commit)
print("Notebook validation files exist:", (repo_dir / "ai-training/notebooks").exists())

In [ ]:
# ===== Install pinned Python dependencies =====
from pathlib import Path
import subprocess
import sys

repo_dir = Path("/content/CueNote")
requirements = repo_dir / "ai-training/requirements-colab.txt"
if not requirements.exists():
    raise FileNotFoundError(f"Missing requirements file: {requirements}")

print("Installing:", requirements)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)

In [ ]:
# ===== Apply small first-run Colab limits to cloned config =====
from pathlib import Path
import json
import os
import shutil

repo_dir = Path("/content/CueNote")
run_mode = os.environ["CUENOTE_RUN_MODE"]

colab_config_path = repo_dir / "ai-training/configs/colab/phase9_colab.json"
colab_config = json.loads(colab_config_path.read_text())
colab_config.setdefault("datasetPolicy", {})["assumedDriveCapacityGb"] = 14
colab_config["datasetPolicy"]["minimumFreeDriveGbBeforeTraining"] = 3
colab_config.setdefault("checkpointPolicy", {})["resumeIfCompatible"] = bool(AUTO_RESUME)
colab_config["datasetPolicy"]["maxDenseImagesForColabSubset"] = int(MAX_DENSE_IMAGES)
colab_config["datasetPolicy"]["maxDenseSourceGroupsForColabSubset"] = int(MAX_DENSE_SOURCE_GROUPS)
if run_mode in {"SYMBOL_TILE_OVERFIT", "SYMBOL_TILE_TRAIN"}:
    policy_key = "tileOverfitPolicy" if run_mode == "SYMBOL_TILE_OVERFIT" else "tileTrainPolicy"
    colab_config.setdefault(policy_key, {})["cropSize"] = int(TILE_CROP_SIZE)
    colab_config[policy_key]["inputSize"] = int(TRAIN_INPUT_SIZE)
    colab_config[policy_key]["batchSize"] = int(TRAIN_BATCH_SIZE)
    colab_config[policy_key]["epochs"] = int(TRAIN_EPOCHS)
    colab_config[policy_key]["overlap"] = float(TILE_OVERLAP)
    colab_config[policy_key]["earlyStoppingPatience"] = int(EARLY_STOPPING_PATIENCE)
colab_config_path.write_text(json.dumps(colab_config, indent=2), encoding="utf-8")

for relative in ["ai-training/configs/layout/yolo_layout_colab.json", "ai-training/configs/symbol/yolo_symbol_colab.json"]:
    config_path = repo_dir / relative
    if config_path.exists():
        config = json.loads(config_path.read_text())
        config["epochs"] = int(TRAIN_EPOCHS)
        config["batchSize"] = int(TRAIN_BATCH_SIZE)
        config["inputSize"] = int(TRAIN_INPUT_SIZE)
        config["earlyStoppingPatience"] = int(EARLY_STOPPING_PATIENCE)
        config["workers"] = min(int(config.get("workers", 2)), 2)
        config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
        print("Updated", config_path)
        print(json.dumps({
            "epochs": config["epochs"],
            "batchSize": config["batchSize"],
            "inputSize": config["inputSize"],
            "earlyStoppingPatience": config["earlyStoppingPatience"],
            "workers": config["workers"],
        }, indent=2))

if CLEAN_SCRATCH_RUNS:
    scratch_root = Path("/content/cuenote-phase9")
    for path in [scratch_root / "runs"]:
        shutil.rmtree(path, ignore_errors=True)
    print("Cleaned scratch run directories under /content/cuenote-phase9/runs")

print("Dataset policy:")
print(json.dumps(colab_config["datasetPolicy"], indent=2))
print("Checkpoint policy:")
print(json.dumps(colab_config.get("checkpointPolicy", {}), indent=2))

In [ ]:
# ===== GPU and storage preflight =====
from pathlib import Path
import os
import shutil
import torch

run_mode = os.environ["CUENOTE_RUN_MODE"]
drive_root = Path(os.environ["CUENOTE_DRIVE_ROOT"])
drive_usage = shutil.disk_usage(str(drive_root))
free_gb = drive_usage.free / (1024 ** 3)

print("Run mode:", run_mode)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2))
elif run_mode != "SMOKE":
    raise RuntimeError("GPU runtime is required for LAYOUT_TRAIN, SYMBOL_TRAIN, and FULL_PIPELINE. Use Runtime -> Change runtime type -> GPU.")

print("Drive free GB:", round(free_gb, 2))
if run_mode != "SMOKE" and free_gb < 3:
    raise RuntimeError("Drive free space is below 3GB. Clean Drive before training.")

In [ ]:
# ===== Run CueNote Phase 9 pipeline =====
from pathlib import Path
import os
import subprocess
import sys

repo_dir = Path("/content/CueNote")
cmd = [
    sys.executable,
    str(repo_dir / "ai-training/python/phase9_colab_entry.py"),
    "--run-mode", os.environ["CUENOTE_RUN_MODE"],
    "--drive-root", os.environ["CUENOTE_DRIVE_ROOT"],
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print("RETURN CODE:", result.returncode)
print("\n===== STDOUT =====")
print(result.stdout[-20000:] if result.stdout else "")
print("\n===== STDERR =====")
print(result.stderr[-20000:] if result.stderr else "")
if result.returncode != 0:
    raise RuntimeError(f"CueNote Phase 9 pipeline failed with return code {result.returncode}. See STDOUT/STDERR above.")

In [ ]:
# ===== Summarize outputs =====
from pathlib import Path
import json
import os

root = Path(os.environ["CUENOTE_DRIVE_ROOT"])
print("Drive root:", root)

run_state = root / "run-state.json"
if run_state.exists():
    print("
run-state.json")
    print(run_state.read_text()[:4000])

print("
Conversion reports")
for path in sorted(root.rglob("conversion-report.json")):
    try:
        report = json.loads(path.read_text())
    except Exception as error:
        print(path, error)
        continue
    print("==", path)
    print(json.dumps({
        "itemCount": report.get("itemCount"),
        "sourceGroupCount": report.get("sourceGroupCount"),
        "maxItems": report.get("maxItems"),
        "maxSourceGroups": report.get("maxSourceGroups"),
        "allowedClassIds": report.get("allowedClassIds"),
        "classIds": report.get("classIds"),
        "excludedAnnotations": report.get("excludedAnnotations"),
        "invalidBoundingBoxes": report.get("invalidBoundingBoxes"),
        "topUnmappedClasses": report.get("topUnmappedClasses"),
        "annotationFile": report.get("annotationFile"),
        "splits": report.get("splits"),
    }, indent=2))

print("
Artifacts")
for path in sorted((root / "artifacts").rglob("*.zip")) if (root / "artifacts").exists() else []:
    print(path)

print("
Checkpoints")
for path in sorted((root / "checkpoints").rglob("*.pt")) if (root / "checkpoints").exists() else []:
    print(path)

## ?? ?? ??

`SMOKE` ???? dataset preparation??? ????. ?? ?? ?? ??? ???.

`LAYOUT_TRAIN` ???? ??? Drive? ??? ??.

- `checkpoints/layout/last.pt`
- `checkpoints/layout/best.pt`
- `reports/*evaluation.json`
- `onnx/*.onnx` ?? model export output
- `artifacts/*.zip`

? 1 epoch, 64 image subset ??? ?? ??? ??? **EXPERIMENTAL smoke-training artifact**?. mAP? ??? 0??? ????? ?? ???? ??? ? ??. ?? ?? ?? ??? ? ? subset? ?? validation/test report ??? ??.